# AgriTrust — Fraud Detector Training
Trains an Isolation Forest on transaction data to detect anomalous/fraudulent activity.

**Output:** `ml_weights/fraud_model_v1.pkl`

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, 'data/training'))
print('Repo root:', REPO_ROOT)

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, precision_score, recall_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 1. Load Transaction Data

In [ ]:
def load_data():
    try:
        from backend.app.db.session import SessionLocal
        from backend.app.models.transaction import Transaction, Order
        db = SessionLocal()
        orders = db.query(Order).all()
        if len(orders) < 20:
            raise ValueError('Not enough orders in DB yet')
        rows = []
        for o in orders:
            rows.append({
                'amount': float(o.total_amount or 0),
                'quantity': float(o.quantity or 0),
                'price_per_unit': float(o.price_per_unit or 0),
                'hour': o.created_at.hour if o.created_at else 12,
                'is_fraud': 1 if o.status and 'fraud' in str(o.status).lower() else 0,
            })
        db.close()
        df = pd.DataFrame(rows)
        print(f'Loaded {len(df)} orders from database')
        return df
    except Exception as e:
        print(f'DB unavailable ({e}) — using synthetic data')
        from utils.data_loader import generate_sample_data
        return generate_sample_data('transactions', n_samples=20000)

df = load_data()
print(f'Fraud rate: {df["is_fraud"].mean()*100:.1f}%')
df.head()

## 2. Feature Engineering

In [ ]:
features = pd.DataFrame()
features['amount']        = df['amount']
features['quantity']      = df['quantity']
features['price_per_kg']  = df['amount'] / (df['quantity'] + 1)
features['hour']          = df['hour']
features['is_off_hour']   = (~df['hour'].between(6, 19)).astype(int)
features['log_amount']    = np.log1p(df['amount'])
features['log_quantity']  = np.log1p(df['quantity'])

# Add farmer/buyer history if available
if 'farmer_history' in df.columns:
    features['farmer_history'] = df['farmer_history']
    features['buyer_history']  = df['buyer_history']

print(f'Features: {list(features.columns)}')

## 3. Scale + Train Isolation Forest

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

model = IsolationForest(
    contamination=0.05,
    random_state=42,
    n_estimators=100
)
model.fit(X_scaled)
print('Training complete')

## 4. Evaluate

In [ ]:
preds = model.predict(X_scaled)
preds_binary = (preds == -1).astype(int)

print(classification_report(df['is_fraud'], preds_binary))
print(f'Anomalies flagged: {preds_binary.sum()} ({preds_binary.mean()*100:.1f}%)')

# Anomaly score distribution
scores = model.score_samples(X_scaled)
plt.figure(figsize=(8, 4))
plt.hist(scores, bins=50, color='steelblue', edgecolor='white')
plt.axvline(x=np.percentile(scores, 5), color='red', linestyle='--', label='5th percentile (fraud threshold)')
plt.title('Anomaly Score Distribution')
plt.xlabel('Score (lower = more anomalous)')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Save

In [ ]:
WEIGHTS = os.path.join(REPO_ROOT, 'ml_weights')
os.makedirs(WEIGHTS, exist_ok=True)
joblib.dump(model,                    f'{WEIGHTS}/fraud_model_v1.pkl')
joblib.dump(scaler,                   f'{WEIGHTS}/fraud_scaler_v1.pkl')
joblib.dump(features.columns.tolist(),f'{WEIGHTS}/fraud_features_v1.pkl')
print('Saved to ml_weights/')